# 03 — Multi-Omics Analysis of Genetic Heterogeneity in Thyroid Cancer (TCGA-THCA)

**Author:** Kurshid  
**Project:** Genetic Heterogeneity in Thyroid Cancer  
**GitHub:** kurshid1991/Genetic-heterogeneity-thyroid-cancer

---

## Pipeline Overview

This notebook performs a **multi-omics integrative analysis** of thyroid carcinoma (TCGA-THCA) 
using **6 data types** from the GDC portal:

| # | Data Type | Category | Expected Files |
|---|-----------|----------|---------------|
| 1 | Somatic Mutations | MAF (Masked Somatic Mutation) | ~498 |
| 2 | Gene Expression | RNA-Seq (HTSeq-FPKM) | ~572 |
| 3 | Copy Number Variation | Masked CNV Segment | ~494 |
| 4 | DNA Methylation | Illumina HM450 | ~567 |
| 5 | miRNA Expression | miRNA-Seq Quantification | ~570 |
| 6 | Clinical Data | BCR XML / JSON | ~507 |

**Total: ~3,200+ files**

### Analysis Steps:
1. Download all data from GDC API
2. Process each omics layer into patient × feature matrices
3. Somatic mutation landscape analysis
4. Differential gene expression (tumor vs. normal)
5. Copy number alteration profiling
6. Multi-omics integration (PCA + clustering)
7. Survival analysis by molecular subtype
8. Publication-quality figures

---

## 0. Setup & Imports

In [ ]:
import os
import sys
import glob
import json
import time
import gzip
import shutil
import requests
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.stats import mannwhitneyu, chi2_contingency

warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})

# Project directories
PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

for d in [DATA_DIR, RESULTS_DIR, FIGURES_DIR,
          DATA_DIR / "maf", DATA_DIR / "expression", 
          DATA_DIR / "cnv", DATA_DIR / "methylation",
          DATA_DIR / "mirna", DATA_DIR / "clinical"]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ All imports successful")
print(f"Project directory: {PROJECT_DIR.resolve()}")

---
## 1. Download All Data from GDC API

We query the GDC API for **6 data types** from TCGA-THCA and download all open-access files.
This is the most time-consuming step (~30-60 min depending on internet speed).

In [ ]:
# ── GDC API Helper Functions ──────────────────────────────────────

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
GDC_DATA_URL  = "https://api.gdc.cancer.gov/data"
GDC_CASES_URL = "https://api.gdc.cancer.gov/cases"

def gdc_query(data_category, data_type, data_format=None, extra_filters=None):
    """Query GDC API for file UUIDs matching criteria."""
    content = [
        {"op": "=", "content": {"field": "cases.project.project_id", "value": "TCGA-THCA"}},
        {"op": "=", "content": {"field": "data_category", "value": data_category}},
        {"op": "=", "content": {"field": "data_type", "value": data_type}},
        {"op": "=", "content": {"field": "access", "value": "open"}},
    ]
    if data_format:
        content.append({"op": "=", "content": {"field": "data_format", "value": data_format}})
    if extra_filters:
        content.extend(extra_filters)
    
    filters = {"op": "and", "content": content}
    
    params = {
        "filters": json.dumps(filters),
        "fields": "file_id,file_name,file_size,cases.case_id,cases.submitter_id",
        "format": "JSON",
        "size": 5000,
    }
    
    resp = requests.get(GDC_FILES_URL, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    hits = data["data"]["hits"]
    total = data["data"]["pagination"]["total"]
    print(f"  Found {total} files for {data_type}")
    return hits


def download_files(file_hits, output_dir, label="files"):
    """Download files from GDC, skipping already-downloaded ones."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    downloaded = 0
    skipped = 0
    errors = 0
    
    for i, hit in enumerate(file_hits):
        file_id = hit["file_id"]
        file_name = hit["file_name"]
        
        # Check if already downloaded (with or without .gz)
        out_path = output_dir / file_name
        out_path_ungz = output_dir / file_name.replace(".gz", "")
        if out_path.exists() or out_path_ungz.exists():
            skipped += 1
            continue
        
        try:
            url = f"{GDC_DATA_URL}/{file_id}"
            resp = requests.get(url, timeout=120, stream=True)
            resp.raise_for_status()
            
            with open(out_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            # Decompress .gz files
            if file_name.endswith(".gz"):
                with gzip.open(out_path, 'rb') as f_in:
                    with open(out_path_ungz, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                out_path.unlink()
            
            downloaded += 1
            
        except Exception as e:
            errors += 1
            if errors <= 3:
                print(f"  ⚠️  Error on {file_name}: {e}")
        
        if (i + 1) % 50 == 0:
            print(f"  Progress: {i+1}/{len(file_hits)} | Downloaded: {downloaded} | Skipped: {skipped}")
    
    total = downloaded + skipped
    print(f"  ✅ {label}: {total} files ready ({downloaded} new, {skipped} existing, {errors} errors)")
    return total

print("✅ GDC API helper functions defined")

In [ ]:
# ── 1a. Somatic Mutations (MAF) ──────────────────────────────────
print("=" * 60)
print("1a. Downloading Somatic Mutation MAF files...")
print("=" * 60)

maf_hits = gdc_query(
    data_category="Simple Nucleotide Variation",
    data_type="Masked Somatic Mutation",
    data_format="MAF"
)
n_maf = download_files(maf_hits, DATA_DIR / "maf", label="MAF files")

In [ ]:
# ── 1b. Gene Expression (RNA-Seq FPKM) ───────────────────────────
print("=" * 60)
print("1b. Downloading Gene Expression (RNA-Seq) files...")
print("=" * 60)

expr_hits = gdc_query(
    data_category="Transcriptome Profiling",
    data_type="Gene Expression Quantification",
    extra_filters=[
        {"op": "=", "content": {"field": "analysis.workflow_type", "value": "STAR - Counts"}}
    ]
)
n_expr = download_files(expr_hits, DATA_DIR / "expression", label="Expression files")

In [ ]:
# ── 1c. Copy Number Variation ────────────────────────────────────
print("=" * 60)
print("1c. Downloading Copy Number Variation files...")
print("=" * 60)

cnv_hits = gdc_query(
    data_category="Copy Number Variation",
    data_type="Masked Copy Number Segment",
)
n_cnv = download_files(cnv_hits, DATA_DIR / "cnv", label="CNV files")

In [ ]:
# ── 1d. DNA Methylation ──────────────────────────────────────────
print("=" * 60)
print("1d. Downloading DNA Methylation files...")
print("=" * 60)

meth_hits = gdc_query(
    data_category="DNA Methylation",
    data_type="Methylation Beta Value",
)
n_meth = download_files(meth_hits, DATA_DIR / "methylation", label="Methylation files")

In [ ]:
# ── 1e. miRNA Expression ─────────────────────────────────────────
print("=" * 60)
print("1e. Downloading miRNA Expression files...")
print("=" * 60)

mirna_hits = gdc_query(
    data_category="Transcriptome Profiling",
    data_type="miRNA Expression Quantification",
)
n_mirna = download_files(mirna_hits, DATA_DIR / "mirna", label="miRNA files")

In [ ]:
# ── 1f. Clinical Data ────────────────────────────────────────────
print("=" * 60)
print("1f. Downloading Clinical Data...")
print("=" * 60)

# Clinical data via cases endpoint
params = {
    "filters": json.dumps({
        "op": "=",
        "content": {"field": "project.project_id", "value": "TCGA-THCA"}
    }),
    "fields": ",".join([
        "submitter_id", "case_id",
        "demographic.gender", "demographic.race", "demographic.ethnicity",
        "demographic.vital_status", "demographic.days_to_death", "demographic.year_of_birth",
        "demographic.age_at_index",
        "diagnoses.primary_diagnosis", "diagnoses.tumor_stage",
        "diagnoses.age_at_diagnosis", "diagnoses.days_to_last_follow_up",
        "diagnoses.morphology", "diagnoses.site_of_resection_or_biopsy",
        "diagnoses.tissue_or_organ_of_origin",
        "diagnoses.classification_of_tumor", "diagnoses.prior_malignancy",
        "diagnoses.ajcc_pathologic_stage", "diagnoses.ajcc_pathologic_t",
        "diagnoses.ajcc_pathologic_n", "diagnoses.ajcc_pathologic_m",
    ]),
    "format": "JSON",
    "size": 1000,
}

resp = requests.get(GDC_CASES_URL, params=params, timeout=60)
resp.raise_for_status()
clinical_raw = resp.json()["data"]["hits"]
print(f"  Found {len(clinical_raw)} cases with clinical data")

# Flatten clinical data
clinical_records = []
for case in clinical_raw:
    rec = {
        "case_id": case.get("case_id"),
        "patient_id": case.get("submitter_id"),
    }
    demo = case.get("demographic", {})
    if demo:
        rec["gender"] = demo.get("gender")
        rec["race"] = demo.get("race")
        rec["vital_status"] = demo.get("vital_status")
        rec["days_to_death"] = demo.get("days_to_death")
        rec["age_at_index"] = demo.get("age_at_index")
    
    diags = case.get("diagnoses", [])
    if diags:
        d = diags[0]
        rec["primary_diagnosis"] = d.get("primary_diagnosis")
        rec["tumor_stage"] = d.get("ajcc_pathologic_stage")
        rec["pathologic_T"] = d.get("ajcc_pathologic_t")
        rec["pathologic_N"] = d.get("ajcc_pathologic_n")
        rec["pathologic_M"] = d.get("ajcc_pathologic_m")
        rec["days_to_last_follow_up"] = d.get("days_to_last_follow_up")
        rec["morphology"] = d.get("morphology")
        rec["age_at_diagnosis"] = d.get("age_at_diagnosis")
    
    clinical_records.append(rec)

clinical_df = pd.DataFrame(clinical_records)
clinical_df.to_csv(DATA_DIR / "clinical" / "clinical_data.csv", index=False)
print(f"  ✅ Clinical data: {clinical_df.shape[0]} patients, {clinical_df.shape[1]} features")
print(f"\nClinical columns: {list(clinical_df.columns)}")

In [ ]:
# ── Download Summary ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("DOWNLOAD SUMMARY")
print("=" * 60)

for dtype, ddir in [
    ("MAF (mutations)", DATA_DIR / "maf"),
    ("Gene Expression", DATA_DIR / "expression"),
    ("Copy Number", DATA_DIR / "cnv"),
    ("Methylation", DATA_DIR / "methylation"),
    ("miRNA", DATA_DIR / "mirna"),
    ("Clinical", DATA_DIR / "clinical"),
]:
    n = len(list(Path(ddir).glob("*"))) if ddir.exists() else 0
    print(f"  {dtype:25s}: {n:>5} files")

total_files = sum(
    len(list(Path(d).glob("*")))
    for d in [DATA_DIR/"maf", DATA_DIR/"expression", DATA_DIR/"cnv", 
              DATA_DIR/"methylation", DATA_DIR/"mirna", DATA_DIR/"clinical"]
    if d.exists()
)
print(f"\n  {'TOTAL':25s}: {total_files:>5} files")
print("=" * 60)

---
## 2. Somatic Mutation Analysis

Process all MAF files into a merged mutation table, build the binary mutation matrix,
and generate the mutation landscape visualization.

In [ ]:
# ── 2a. Merge all MAF files ──────────────────────────────────────
print("Merging MAF files...")

KEEP_COLS = [
    'Hugo_Symbol', 'Chromosome', 'Start_Position', 'End_Position',
    'Variant_Classification', 'Variant_Type', 'Reference_Allele',
    'Tumor_Seq_Allele2', 'Tumor_Sample_Barcode',
    'HGVSc', 'HGVSp_Short', 'IMPACT', 'SIFT', 'PolyPhen'
]

maf_files = list((DATA_DIR / "maf").glob("*.maf*"))
print(f"Found {len(maf_files)} MAF files")

frames = []
errors = []
for i, fp in enumerate(maf_files):
    try:
        df = pd.read_csv(fp, sep='\t', comment='#', low_memory=False)
        available = [c for c in KEEP_COLS if c in df.columns]
        frames.append(df[available])
    except Exception as e:
        errors.append((fp.name, str(e)))
    if (i+1) % 100 == 0:
        print(f"  Processed {i+1}/{len(maf_files)}...")

merged_maf = pd.concat(frames, ignore_index=True)
merged_maf['Patient_ID'] = merged_maf['Tumor_Sample_Barcode'].str[:12]

merged_maf.to_csv(DATA_DIR / "merged_maf.tsv", sep='\t', index=False)
print(f"\n✅ Merged MAF: {merged_maf.shape[0]:,} mutations across {merged_maf['Patient_ID'].nunique()} patients")
print(f"   Unique genes: {merged_maf['Hugo_Symbol'].nunique()}")
if errors:
    print(f"   Errors: {len(errors)}")

In [ ]:
# ── 2b. Build binary mutation matrix (freq >= 2) ─────────────────
mutation_matrix = (
    merged_maf.groupby(['Patient_ID', 'Hugo_Symbol']).size()
    .unstack(fill_value=0)
)
mutation_matrix = (mutation_matrix > 0).astype(int)

gene_freq = mutation_matrix.sum(axis=0)
genes_keep = gene_freq[gene_freq >= 2].index
mut_matrix = mutation_matrix[genes_keep]

print(f"Full matrix: {mutation_matrix.shape[0]} patients × {mutation_matrix.shape[1]} genes")
print(f"Filtered (freq≥2): {mut_matrix.shape[0]} patients × {mut_matrix.shape[1]} genes")

# Save
mut_matrix.to_csv(RESULTS_DIR / "mutation_matrix.tsv", sep='\t')
print("\n✅ Saved mutation_matrix.tsv")

In [ ]:
# ── 2c. Mutation landscape — Top mutated genes ───────────────────
top_n = 30
top_genes = gene_freq[genes_keep].sort_values(ascending=False).head(top_n)

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, top_n))
bars = ax.barh(range(top_n), top_genes.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_genes.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Number of Patients with Mutation", fontsize=11)
ax.set_title(f"Top {top_n} Most Frequently Mutated Genes\nTCGA-THCA (n={mut_matrix.shape[0]} patients)", fontsize=13)

# Add count labels
for i, v in enumerate(top_genes.values):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig1_top_mutated_genes.png", bbox_inches='tight')
plt.show()
print("✅ Saved fig1_top_mutated_genes.png")

In [ ]:
# ── 2d. Variant classification breakdown ─────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: variant classification counts
vc = merged_maf['Variant_Classification'].value_counts().head(10)
ax1.barh(range(len(vc)), vc.values, color=sns.color_palette("mako", len(vc)))
ax1.set_yticks(range(len(vc)))
ax1.set_yticklabels(vc.index, fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel("Count")
ax1.set_title("Variant Classification Distribution")

# Right: variant type pie
if 'Variant_Type' in merged_maf.columns:
    vt = merged_maf['Variant_Type'].value_counts()
    ax2.pie(vt.values, labels=vt.index, autopct='%1.1f%%', 
            colors=sns.color_palette("Set2", len(vt)), startangle=90)
    ax2.set_title("Variant Types")

plt.suptitle("Mutation Characteristics — TCGA-THCA", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig2_variant_classification.png", bbox_inches='tight')
plt.show()
print("✅ Saved fig2_variant_classification.png")

In [ ]:
# ── 2e. IMPACT distribution ──────────────────────────────────────
if 'IMPACT' in merged_maf.columns:
    impact_order = ['HIGH', 'MODERATE', 'LOW', 'MODIFIER']
    impact_counts = merged_maf['IMPACT'].value_counts()
    impact_counts = impact_counts.reindex([i for i in impact_order if i in impact_counts.index])
    
    fig, ax = plt.subplots(figsize=(8, 5))
    impact_colors = {'HIGH': '#d62728', 'MODERATE': '#ff7f0e', 'LOW': '#2ca02c', 'MODIFIER': '#7f7f7f'}
    bars = ax.bar(impact_counts.index, impact_counts.values, 
                  color=[impact_colors.get(x, '#999') for x in impact_counts.index],
                  edgecolor='white', linewidth=0.5)
    ax.set_ylabel("Number of Mutations")
    ax.set_title("Mutation Impact Distribution — TCGA-THCA")
    for bar, val in zip(bars, impact_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f"{val:,}", ha='center', fontsize=10)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig3_impact_distribution.png", bbox_inches='tight')
    plt.show()
    print("✅ Saved fig3_impact_distribution.png")

---
## 3. Gene Expression Analysis (RNA-Seq)

Process STAR-Counts RNA-Seq files, build expression matrix, and perform
differential expression analysis between tumor and normal samples.

In [ ]:
# ── 3a. Build gene expression matrix ─────────────────────────────
print("Processing gene expression files...")

expr_files = list((DATA_DIR / "expression").glob("*.tsv*")) + \
             list((DATA_DIR / "expression").glob("*.txt*")) + \
             list((DATA_DIR / "expression").glob("*.counts*"))

print(f"Found {len(expr_files)} expression files")

expr_data = {}
gene_names = None

for i, fp in enumerate(expr_files):
    try:
        # STAR-Counts format: gene_id, gene_name, gene_type, unstranded, ...
        df = pd.read_csv(fp, sep='\t', comment='#', skiprows=1 if 'N_' not in open(fp).readline() else 0)
        
        # Try to identify the right columns
        if 'gene_name' in df.columns:
            # New STAR-Counts format
            if 'tpm_unstranded' in df.columns:
                col = 'tpm_unstranded'
            elif 'fpkm_unstranded' in df.columns:
                col = 'fpkm_unstranded'
            elif 'unstranded' in df.columns:
                col = 'unstranded'
            else:
                col = df.columns[3]  # Usually the count column
            
            series = df.set_index('gene_name')[col]
        elif df.shape[1] == 2:
            # Simple gene_id 	 count format
            series = df.set_index(df.columns[0])[df.columns[1]]
        else:
            continue
        
        # Extract sample barcode from filename or first few rows
        sample_id = fp.stem.split('.')[0][:16]  # TCGA barcode
        expr_data[sample_id] = series
        
        if gene_names is None:
            gene_names = series.index
            
    except Exception as e:
        pass
    
    if (i+1) % 100 == 0:
        print(f"  Processed {i+1}/{len(expr_files)}...")

if expr_data:
    expr_matrix = pd.DataFrame(expr_data).T
    expr_matrix = expr_matrix.apply(pd.to_numeric, errors='coerce').fillna(0)
    
    # Map to patient IDs
    expr_matrix.index = [idx[:12] if len(idx) >= 12 else idx for idx in expr_matrix.index]
    
    # Remove zero-variance genes
    nonzero = expr_matrix.columns[expr_matrix.var() > 0]
    expr_matrix = expr_matrix[nonzero]
    
    print(f"\n✅ Expression matrix: {expr_matrix.shape[0]} samples × {expr_matrix.shape[1]} genes")
    expr_matrix.to_csv(RESULTS_DIR / "expression_matrix.tsv", sep='\t')
else:
    print("⚠️  No expression files could be parsed. Check file format.")
    expr_matrix = pd.DataFrame()

In [ ]:
# ── 3b. Top variable genes heatmap ───────────────────────────────
if not expr_matrix.empty:
    # Log2 transform (add 1 to avoid log(0))
    expr_log = np.log2(expr_matrix + 1)
    
    # Top 50 most variable genes
    gene_var = expr_log.var().sort_values(ascending=False)
    top_var_genes = gene_var.head(50).index
    
    fig, ax = plt.subplots(figsize=(16, 10))
    sns.heatmap(
        expr_log[top_var_genes].T,
        cmap='RdBu_r', center=0,
        xticklabels=False,
        yticklabels=True,
        ax=ax,
        cbar_kws={'label': 'log2(FPKM+1)'}
    )
    ax.set_title("Top 50 Most Variable Genes — TCGA-THCA Expression", fontsize=13)
    ax.set_xlabel(f"Samples (n={expr_log.shape[0]})")
    ax.set_ylabel("Genes")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig4_expression_heatmap.png", bbox_inches='tight')
    plt.show()
    print("✅ Saved fig4_expression_heatmap.png")

---
## 4. Copy Number Variation (CNV) Analysis

Process masked copy number segment files to identify recurrent amplifications 
and deletions across the cohort.

In [ ]:
# ── 4a. Process CNV segment files ────────────────────────────────
print("Processing CNV segment files...")

cnv_files = list((DATA_DIR / "cnv").glob("*.seg*")) + \
            list((DATA_DIR / "cnv").glob("*.tsv*")) + \
            list((DATA_DIR / "cnv").glob("*.txt*"))

print(f"Found {len(cnv_files)} CNV files")

cnv_frames = []
for i, fp in enumerate(cnv_files):
    try:
        df = pd.read_csv(fp, sep='\t', comment='#', low_memory=False)
        cnv_frames.append(df)
    except:
        pass
    if (i+1) % 100 == 0:
        print(f"  Processed {i+1}/{len(cnv_files)}...")

if cnv_frames:
    cnv_all = pd.concat(cnv_frames, ignore_index=True)
    
    # Extract patient ID from sample barcode
    barcode_col = [c for c in cnv_all.columns if 'barcode' in c.lower() or 'sample' in c.lower()]
    if barcode_col:
        cnv_all['Patient_ID'] = cnv_all[barcode_col[0]].str[:12]
    elif 'GDC_Aliquot' in cnv_all.columns:
        cnv_all['Patient_ID'] = cnv_all['GDC_Aliquot'].str[:12]
    
    print(f"\n✅ CNV data: {cnv_all.shape[0]:,} segments across {cnv_all['Patient_ID'].nunique()} patients")
    print(f"   Columns: {list(cnv_all.columns)}")
    cnv_all.to_csv(RESULTS_DIR / "cnv_segments.tsv", sep='\t', index=False)
else:
    print("⚠️  No CNV files parsed")
    cnv_all = pd.DataFrame()

In [ ]:
# ── 4b. CNV landscape — genome-wide segment mean plot ────────────
if not cnv_all.empty:
    seg_col = [c for c in cnv_all.columns if 'segment_mean' in c.lower() or 'seg' in c.lower()]
    
    if seg_col:
        seg_mean_col = seg_col[0]
        chr_col = [c for c in cnv_all.columns if 'chrom' in c.lower() or 'chr' in c.lower()][0]
        
        fig, ax = plt.subplots(figsize=(16, 5))
        
        # Classify: amplification (>0.3) vs deletion (<-0.3)
        cnv_all['CNV_status'] = 'Neutral'
        cnv_all.loc[cnv_all[seg_mean_col] > 0.3, 'CNV_status'] = 'Amplification'
        cnv_all.loc[cnv_all[seg_mean_col] < -0.3, 'CNV_status'] = 'Deletion'
        
        cnv_summary = cnv_all['CNV_status'].value_counts()
        colors = {'Amplification': '#d62728', 'Deletion': '#1f77b4', 'Neutral': '#cccccc'}
        
        ax.bar(cnv_summary.index, cnv_summary.values,
               color=[colors[x] for x in cnv_summary.index])
        ax.set_ylabel("Number of Segments")
        ax.set_title("Copy Number Alteration Summary — TCGA-THCA")
        
        for i, (idx, val) in enumerate(cnv_summary.items()):
            ax.text(i, val + 100, f"{val:,}", ha='center', fontsize=10)
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "fig5_cnv_summary.png", bbox_inches='tight')
        plt.show()
        print("✅ Saved fig5_cnv_summary.png")

---
## 5. DNA Methylation Analysis

Process Illumina HM450 methylation beta values. Due to large file sizes,
we extract gene-level methylation summaries.

In [ ]:
# ── 5a. Process methylation files (sample from first N probes) ───
print("Processing methylation files...")

meth_files = list((DATA_DIR / "methylation").glob("*"))
print(f"Found {len(meth_files)} methylation files")

# Methylation files can be very large — sample strategically
meth_data = {}
n_processed = 0

for i, fp in enumerate(meth_files):
    try:
        # Read first column (probe) and beta value
        df = pd.read_csv(fp, sep='\t', nrows=None, comment='#', low_memory=False)
        
        # Typical columns: Composite Element REF, Beta_value
        if df.shape[1] >= 2:
            probe_col = df.columns[0]
            beta_col = [c for c in df.columns if 'beta' in c.lower()]
            if not beta_col:
                beta_col = [df.columns[1]]
            
            series = pd.to_numeric(df.set_index(probe_col)[beta_col[0]], errors='coerce')
            sample_id = fp.stem[:16]
            meth_data[sample_id] = series
            n_processed += 1
    except:
        pass
    
    if (i+1) % 50 == 0:
        print(f"  Processed {i+1}/{len(meth_files)}...")

if meth_data:
    meth_matrix = pd.DataFrame(meth_data).T
    meth_matrix.index = [idx[:12] for idx in meth_matrix.index]
    print(f"\n✅ Methylation matrix: {meth_matrix.shape[0]} samples × {meth_matrix.shape[1]} probes")
    
    # Save a summary (top variable probes)
    probe_var = meth_matrix.var().sort_values(ascending=False)
    top_probes = probe_var.head(5000).index
    meth_matrix[top_probes].to_csv(RESULTS_DIR / "methylation_matrix_top5k.tsv", sep='\t')
    print(f"   Saved top 5000 variable probes")
else:
    print("⚠️  No methylation files parsed")
    meth_matrix = pd.DataFrame()

---
## 6. miRNA Expression Analysis

Process miRNA quantification files and identify top differentially expressed miRNAs.

In [ ]:
# ── 6a. Process miRNA expression files ───────────────────────────
print("Processing miRNA expression files...")

mirna_files = list((DATA_DIR / "mirna").glob("*"))
print(f"Found {len(mirna_files)} miRNA files")

mirna_data = {}
for i, fp in enumerate(mirna_files):
    try:
        df = pd.read_csv(fp, sep='\t', comment='#')
        
        # Typical: miRNA_ID, read_count, reads_per_million_miRNA_mapped
        if 'miRNA_ID' in df.columns:
            rpm_col = [c for c in df.columns if 'per_million' in c.lower()]
            if rpm_col:
                series = df.set_index('miRNA_ID')[rpm_col[0]]
            else:
                series = df.set_index('miRNA_ID')[df.columns[1]]
            
            sample_id = fp.stem[:16]
            mirna_data[sample_id] = series
    except:
        pass
    
    if (i+1) % 100 == 0:
        print(f"  Processed {i+1}/{len(mirna_files)}...")

if mirna_data:
    mirna_matrix = pd.DataFrame(mirna_data).T
    mirna_matrix = mirna_matrix.apply(pd.to_numeric, errors='coerce').fillna(0)
    mirna_matrix.index = [idx[:12] for idx in mirna_matrix.index]
    
    print(f"\n✅ miRNA matrix: {mirna_matrix.shape[0]} samples × {mirna_matrix.shape[1]} miRNAs")
    mirna_matrix.to_csv(RESULTS_DIR / "mirna_matrix.tsv", sep='\t')
else:
    print("⚠️  No miRNA files parsed")
    mirna_matrix = pd.DataFrame()

---
## 7. Clinical Data Exploration

Visualize demographics, staging, and survival data from the TCGA-THCA cohort.

In [ ]:
# ── 7a. Clinical demographics ────────────────────────────────────
clinical = pd.read_csv(DATA_DIR / "clinical" / "clinical_data.csv")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender
if 'gender' in clinical.columns:
    gender_counts = clinical['gender'].value_counts()
    axes[0,0].pie(gender_counts.values, labels=gender_counts.index, 
                  autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'], startangle=90)
    axes[0,0].set_title("Gender Distribution")

# Age at diagnosis
if 'age_at_diagnosis' in clinical.columns:
    age_years = clinical['age_at_diagnosis'].dropna() / 365.25
    axes[0,1].hist(age_years, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0,1].set_xlabel("Age at Diagnosis (years)")
    axes[0,1].set_ylabel("Count")
    axes[0,1].set_title(f"Age Distribution (median={age_years.median():.0f} yrs)")
    axes[0,1].axvline(age_years.median(), color='red', linestyle='--', alpha=0.7)

# Tumor stage
if 'tumor_stage' in clinical.columns:
    stage_counts = clinical['tumor_stage'].dropna().value_counts().head(8)
    axes[1,0].barh(range(len(stage_counts)), stage_counts.values, 
                   color=sns.color_palette("YlOrRd", len(stage_counts)))
    axes[1,0].set_yticks(range(len(stage_counts)))
    axes[1,0].set_yticklabels(stage_counts.index, fontsize=9)
    axes[1,0].invert_yaxis()
    axes[1,0].set_xlabel("Count")
    axes[1,0].set_title("AJCC Pathologic Stage")

# Vital status
if 'vital_status' in clinical.columns:
    vs_counts = clinical['vital_status'].value_counts()
    axes[1,1].pie(vs_counts.values, labels=vs_counts.index, 
                  autopct='%1.1f%%', colors=['#2ca02c', '#d62728'], startangle=90)
    axes[1,1].set_title("Vital Status")

plt.suptitle("TCGA-THCA Clinical Overview", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig6_clinical_overview.png", bbox_inches='tight')
plt.show()
print("✅ Saved fig6_clinical_overview.png")

---
## 8. Multi-Omics Integration — PCA + Clustering

Integrate mutation, expression, CNV, and miRNA data for joint dimensionality
reduction and unsupervised clustering to identify molecular subtypes.

In [ ]:
# ── 8a. Identify shared patients across omics layers ─────────────
print("Finding patients shared across omics layers...")

available_omics = {}
if not mut_matrix.empty:
    available_omics['Mutation'] = set(mut_matrix.index)
if 'expr_matrix' in dir() and not expr_matrix.empty:
    available_omics['Expression'] = set(expr_matrix.index)
if 'mirna_matrix' in dir() and not mirna_matrix.empty:
    available_omics['miRNA'] = set(mirna_matrix.index)
if 'meth_matrix' in dir() and not meth_matrix.empty:
    available_omics['Methylation'] = set(meth_matrix.index)

print(f"\nAvailable omics layers: {list(available_omics.keys())}")
for name, pats in available_omics.items():
    print(f"  {name:15s}: {len(pats)} patients")

# Find intersection
if len(available_omics) >= 2:
    shared_patients = sorted(set.intersection(*available_omics.values()))
    print(f"\nShared across all layers: {len(shared_patients)} patients")
else:
    shared_patients = sorted(list(available_omics.values())[0]) if available_omics else []
    print(f"\nUsing single layer: {len(shared_patients)} patients")

In [ ]:
# ── 8b. Build integrated feature matrix ──────────────────────────
print("Building integrated feature matrix...")

feature_blocks = []
feature_labels = []

# Mutation features (binary, top 200 genes)
if 'Mutation' in available_omics and shared_patients:
    mut_sub = mut_matrix.loc[mut_matrix.index.isin(shared_patients)]
    top_mut_genes = mut_sub.sum().sort_values(ascending=False).head(200).index
    mut_features = mut_sub[top_mut_genes]
    mut_features = mut_features.loc[shared_patients]
    feature_blocks.append(mut_features.values)
    feature_labels.extend([f"MUT_{g}" for g in top_mut_genes])
    print(f"  Mutation: {mut_features.shape[1]} features")

# Expression features (top 500 variable genes, log-scaled)
if 'Expression' in available_omics and shared_patients:
    expr_sub = expr_matrix.loc[expr_matrix.index.isin(shared_patients)]
    expr_log = np.log2(expr_sub + 1)
    top_expr = expr_log.var().sort_values(ascending=False).head(500).index
    expr_features = expr_log[top_expr]
    expr_features = expr_features.loc[shared_patients]
    feature_blocks.append(StandardScaler().fit_transform(expr_features))
    feature_labels.extend([f"EXPR_{g}" for g in top_expr])
    print(f"  Expression: {expr_features.shape[1]} features")

# miRNA features (all, log-scaled)
if 'miRNA' in available_omics and shared_patients:
    mirna_sub = mirna_matrix.loc[mirna_matrix.index.isin(shared_patients)]
    mirna_log = np.log2(mirna_sub + 1)
    top_mirna = mirna_log.var().sort_values(ascending=False).head(200).index
    mirna_features = mirna_log[top_mirna]
    mirna_features = mirna_features.loc[shared_patients]
    feature_blocks.append(StandardScaler().fit_transform(mirna_features))
    feature_labels.extend([f"miRNA_{m}" for m in top_mirna])
    print(f"  miRNA: {mirna_features.shape[1]} features")

# Methylation features (top variable probes)
if 'Methylation' in available_omics and shared_patients:
    meth_sub = meth_matrix.loc[meth_matrix.index.isin(shared_patients)]
    top_meth = meth_sub.var().sort_values(ascending=False).head(500).index
    meth_features = meth_sub[top_meth]
    meth_features = meth_features.loc[shared_patients]
    feature_blocks.append(StandardScaler().fit_transform(meth_features.fillna(0)))
    feature_labels.extend([f"METH_{p}" for p in top_meth])
    print(f"  Methylation: {meth_features.shape[1]} features")

# Concatenate all feature blocks
if feature_blocks:
    X_integrated = np.hstack(feature_blocks)
    print(f"\n✅ Integrated matrix: {X_integrated.shape[0]} patients × {X_integrated.shape[1]} features")
    print(f"   Omics layers used: {len(feature_blocks)}")
else:
    print("⚠️  No feature blocks available")
    X_integrated = np.array([])

In [ ]:
# ── 8c. PCA on integrated data ───────────────────────────────────
if X_integrated.size > 0:
    # Handle NaNs
    X_clean = np.nan_to_num(X_integrated, nan=0.0)
    
    n_comp = min(50, *X_clean.shape)
    pca = PCA(n_components=n_comp, random_state=42)
    X_pca = pca.fit_transform(X_clean)
    
    var_exp = pca.explained_variance_ratio_
    cumvar = np.cumsum(var_exp)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.bar(range(1, 21), var_exp[:20], color='steelblue', alpha=0.8)
    ax1.set_xlabel("Principal Component")
    ax1.set_ylabel("Variance Explained")
    ax1.set_title("Scree Plot — Multi-Omics PCA")
    
    ax2.plot(range(1, n_comp+1), cumvar, 'o-', markersize=3, color='steelblue')
    ax2.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='80%')
    ax2.axhline(0.9, color='orange', linestyle='--', alpha=0.5, label='90%')
    ax2.set_xlabel("Number of Components")
    ax2.set_ylabel("Cumulative Variance")
    ax2.set_title("Cumulative Variance Explained")
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig7_pca_variance.png", bbox_inches='tight')
    plt.show()
    
    n_80 = int(np.argmax(cumvar >= 0.8)) + 1
    print(f"PC1: {var_exp[0]*100:.1f}%, PC2: {var_exp[1]*100:.1f}%")
    print(f"Components for 80% variance: {n_80}")
    print("✅ Saved fig7_pca_variance.png")

In [ ]:
# ── 8d. KMeans — Elbow + Silhouette ──────────────────────────────
if X_integrated.size > 0:
    n_pca_use = max(5, min(20, n_80))
    X_clust = X_pca[:, :n_pca_use]
    print(f"Clustering with {n_pca_use} PCA components")
    
    K_range = range(2, 11)
    inertias, silhouettes = [], []
    
    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=20, max_iter=500)
        labels = km.fit_predict(X_clust)
        inertias.append(km.inertia_)
        sil = silhouette_score(X_clust, labels)
        silhouettes.append(sil)
        print(f"  k={k}: silhouette={sil:.4f}")
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(K_range, inertias, 'bo-', markersize=6)
    ax1.set_xlabel("k")
    ax1.set_ylabel("Inertia")
    ax1.set_title("Elbow Method")
    
    ax2.plot(K_range, silhouettes, 'ro-', markersize=6)
    ax2.set_xlabel("k")
    ax2.set_ylabel("Silhouette Score")
    ax2.set_title("Silhouette Analysis")
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig8_elbow_silhouette.png", bbox_inches='tight')
    plt.show()
    
    best_k = list(K_range)[np.argmax(silhouettes)]
    print(f"\n✅ Optimal k = {best_k}")
    print("✅ Saved fig8_elbow_silhouette.png")

In [ ]:
# ── 8e. Final clustering + PCA scatter ───────────────────────────
if X_integrated.size > 0:
    km_final = KMeans(n_clusters=best_k, random_state=42, n_init=50, max_iter=1000)
    cluster_labels = km_final.fit_predict(X_clust)
    
    cluster_df = pd.DataFrame({
        'Patient_ID': shared_patients,
        'Cluster': cluster_labels,
        'PC1': X_pca[:, 0],
        'PC2': X_pca[:, 1],
    })
    
    print(f"Cluster distribution (k={best_k}):")
    print(cluster_df['Cluster'].value_counts().sort_index())
    
    # PCA scatter
    fig, ax = plt.subplots(figsize=(10, 8))
    palette = sns.color_palette("Set2", best_k)
    
    for i in range(best_k):
        mask = cluster_labels == i
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=[palette[i]], label=f'Cluster {i} (n={mask.sum()})',
                   alpha=0.6, s=50, edgecolors='white', linewidth=0.5)
    
    ax.set_xlabel(f"PC1 ({var_exp[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_exp[1]*100:.1f}%)")
    ax.set_title(f"Multi-Omics PCA — TCGA-THCA (k={best_k})", fontsize=13)
    ax.legend(fontsize=9, framealpha=0.9)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig9_pca_clusters.png", bbox_inches='tight')
    plt.show()
    
    cluster_df.to_csv(RESULTS_DIR / "cluster_assignments.csv", index=False)
    print("✅ Saved fig9_pca_clusters.png")
    print("✅ Saved cluster_assignments.csv")

## 9. Mutation Clustermap — Top Genes by Molecular Subtype

In [ ]:
if X_integrated.size > 0 and not mut_matrix.empty:
    top_hm_genes = 40
    hm_genes = gene_freq[genes_keep].sort_values(ascending=False).head(top_hm_genes).index
    
    hm_data = mut_matrix.loc[mut_matrix.index.isin(shared_patients), hm_genes].copy()
    hm_data['Cluster'] = cluster_df.set_index('Patient_ID')['Cluster']
    hm_data = hm_data.dropna(subset=['Cluster']).sort_values('Cluster')
    
    cluster_for_color = hm_data['Cluster'].astype(int)
    hm_data = hm_data.drop('Cluster', axis=1)
    
    cluster_palette = dict(zip(range(best_k), sns.color_palette("Set2", best_k)))
    row_colors = cluster_for_color.map(cluster_palette)
    
    g = sns.clustermap(
        hm_data, row_cluster=False, col_cluster=True,
        cmap='YlOrRd', figsize=(16, 10),
        row_colors=row_colors.values, yticklabels=False,
        cbar_kws={'label': 'Mutation (0/1)'}, linewidths=0,
    )
    g.fig.suptitle(f"Mutation Landscape — Top {top_hm_genes} Genes by Cluster", y=1.02, fontsize=13)
    
    from matplotlib.patches import Patch
    legend_els = [Patch(facecolor=cluster_palette[i], label=f'Cluster {i}') for i in range(best_k)]
    g.ax_heatmap.legend(handles=legend_els, loc='upper right', bbox_to_anchor=(1.15, 1.05), fontsize=8)
    
    plt.savefig(FIGURES_DIR / "fig10_mutation_clustermap.png", bbox_inches='tight')
    plt.show()
    print("✅ Saved fig10_mutation_clustermap.png")

## 10. Cluster Characterization

In [ ]:
# ── 10a. Top differentially mutated genes per cluster ─────────────
if X_integrated.size > 0 and not mut_matrix.empty:
    mut_with_cluster = mut_matrix.loc[mut_matrix.index.isin(shared_patients)].copy()
    mut_with_cluster['Cluster'] = cluster_df.set_index('Patient_ID')['Cluster']
    mut_with_cluster = mut_with_cluster.dropna(subset=['Cluster'])
    
    print("=" * 70)
    print("CLUSTER CHARACTERIZATION — Differentially Mutated Genes")
    print("=" * 70)
    
    for c in range(best_k):
        in_c = mut_with_cluster[mut_with_cluster['Cluster'] == c].drop('Cluster', axis=1)
        out_c = mut_with_cluster[mut_with_cluster['Cluster'] != c].drop('Cluster', axis=1)
        
        freq_in = in_c.mean()
        freq_out = out_c.mean()
        diff = freq_in - freq_out
        top_enriched = diff.sort_values(ascending=False).head(8)
        
        print(f"\n--- Cluster {c} (n={len(in_c)}) ---")
        print(f"{'Gene':<15} {'In Cluster':>12} {'Outside':>10} {'Diff':>8}")
        print("-" * 48)
        for gene in top_enriched.index:
            print(f"{gene:<15} {freq_in[gene]:>11.1%} {freq_out[gene]:>9.1%} {diff[gene]:>+7.1%}")

In [ ]:
# ── 10b. Clinical features by cluster ────────────────────────────
if X_integrated.size > 0:
    cluster_clinical = clinical.merge(cluster_df[['Patient_ID', 'Cluster']], 
                                       left_on='patient_id', right_on='Patient_ID', how='inner')
    
    if not cluster_clinical.empty:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Stage by cluster
        if 'tumor_stage' in cluster_clinical.columns:
            ct = pd.crosstab(cluster_clinical['Cluster'], cluster_clinical['tumor_stage'])
            ct_norm = ct.div(ct.sum(axis=1), axis=0)
            ct_norm.plot(kind='bar', stacked=True, ax=axes[0], colormap='YlOrRd')
            axes[0].set_title("Tumor Stage by Cluster")
            axes[0].set_ylabel("Proportion")
            axes[0].legend(fontsize=7, bbox_to_anchor=(1.0, 1.0))
        
        # Age by cluster
        if 'age_at_diagnosis' in cluster_clinical.columns:
            age_data = cluster_clinical.dropna(subset=['age_at_diagnosis'])
            age_data['age_years'] = age_data['age_at_diagnosis'] / 365.25
            for c in range(best_k):
                subset = age_data[age_data['Cluster'] == c]['age_years']
                axes[1].hist(subset, bins=20, alpha=0.5, label=f'Cluster {c}')
            axes[1].set_xlabel("Age (years)")
            axes[1].set_title("Age Distribution by Cluster")
            axes[1].legend()
        
        # Gender by cluster
        if 'gender' in cluster_clinical.columns:
            ct = pd.crosstab(cluster_clinical['Cluster'], cluster_clinical['gender'])
            ct.plot(kind='bar', ax=axes[2], color=['#ff9999', '#66b3ff'])
            axes[2].set_title("Gender by Cluster")
            axes[2].set_ylabel("Count")
        
        plt.suptitle("Clinical Features by Molecular Subtype", fontsize=14, y=1.02)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "fig11_clinical_by_cluster.png", bbox_inches='tight')
        plt.show()
        print("✅ Saved fig11_clinical_by_cluster.png")

---
## 11. Survival Analysis by Molecular Subtype

Kaplan-Meier survival curves stratified by cluster assignment.

In [ ]:
# Simple KM without lifelines (avoid extra dependency)
if X_integrated.size > 0 and 'days_to_last_follow_up' in clinical.columns:
    surv = cluster_clinical.copy()
    
    # Survival time: days_to_death if dead, days_to_last_follow_up if alive
    surv['time'] = surv.apply(
        lambda r: r['days_to_death'] if pd.notna(r.get('days_to_death')) 
        else r.get('days_to_last_follow_up', np.nan), axis=1
    )
    surv['event'] = (surv['vital_status'] == 'Dead').astype(int)
    surv = surv.dropna(subset=['time'])
    surv['time'] = surv['time'].astype(float)
    surv = surv[surv['time'] > 0]
    
    if len(surv) > 10:
        fig, ax = plt.subplots(figsize=(10, 7))
        palette = sns.color_palette("Set2", best_k)
        
        for c in range(best_k):
            sub = surv[surv['Cluster'] == c].sort_values('time')
            if len(sub) < 3:
                continue
            
            # Simple KM estimator
            times = sorted(sub['time'].unique())
            n_at_risk = len(sub)
            survival = 1.0
            km_times = [0]
            km_surv = [1.0]
            
            for t in times:
                events_at_t = sub[(sub['time'] == t) & (sub['event'] == 1)].shape[0]
                censored_at_t = sub[(sub['time'] == t) & (sub['event'] == 0)].shape[0]
                
                if n_at_risk > 0:
                    survival *= (1 - events_at_t / n_at_risk)
                
                km_times.append(t)
                km_surv.append(survival)
                n_at_risk -= (events_at_t + censored_at_t)
            
            ax.step(np.array(km_times)/365.25, km_surv, where='post',
                    color=palette[c], linewidth=2,
                    label=f'Cluster {c} (n={len(sub)})')
        
        ax.set_xlabel("Time (years)", fontsize=12)
        ax.set_ylabel("Survival Probability", fontsize=12)
        ax.set_title("Kaplan-Meier Survival by Molecular Subtype — TCGA-THCA", fontsize=13)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / "fig12_survival_curves.png", bbox_inches='tight')
        plt.show()
        print("✅ Saved fig12_survival_curves.png")
    else:
        print("⚠️  Not enough survival data for KM plot")
else:
    print("⚠️  Survival data not available")

---
## 12. Pipeline Summary & Export

In [ ]:
print("\n" + "=" * 70)
print("MULTI-OMICS PIPELINE SUMMARY — TCGA-THCA")
print("=" * 70)

print(f"\n📊 Data Sources:")
for dtype, ddir in [
    ("MAF (Somatic Mutations)", DATA_DIR / "maf"),
    ("Gene Expression (RNA-Seq)", DATA_DIR / "expression"),
    ("Copy Number Variation", DATA_DIR / "cnv"),
    ("DNA Methylation", DATA_DIR / "methylation"),
    ("miRNA Expression", DATA_DIR / "mirna"),
    ("Clinical Data", DATA_DIR / "clinical"),
]:
    n = len(list(Path(ddir).glob("*"))) if ddir.exists() else 0
    print(f"  {dtype:35s}: {n:>5} files")

print(f"\n🧬 Mutation Analysis:")
if not mut_matrix.empty:
    print(f"  Total mutations:    {len(merged_maf):,}")
    print(f"  Patients:           {mut_matrix.shape[0]}")
    print(f"  Genes (freq ≥ 2):   {mut_matrix.shape[1]}")

print(f"\n📈 Multi-Omics Integration:")
if X_integrated.size > 0:
    print(f"  Shared patients:    {len(shared_patients)}")
    print(f"  Total features:     {X_integrated.shape[1]}")
    print(f"  Omics layers:       {len(feature_blocks)}")
    print(f"  Optimal clusters:   {best_k}")
    for c in range(best_k):
        n = (cluster_labels == c).sum()
        print(f"    Cluster {c}: {n} patients ({n/len(cluster_labels)*100:.1f}%)")

print(f"\n📁 Generated Figures:")
for f in sorted(FIGURES_DIR.glob("*.png")):
    print(f"  {f.name}")

print(f"\n📁 Result Files:")
for f in sorted(RESULTS_DIR.glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} MB)")

print(f"\n{'='*70}")
print("🎉 Pipeline complete!")
print("\nNext steps:")
print("  1. git add figures/ results/ notebooks/03_fix_clustering.ipynb")
print("  2. git commit -m 'Multi-omics analysis with corrected clustering'")
print("  3. git push origin main")
print("  4. Update Upwork PDF with new figures")
print("=" * 70)